# Phase 7 — Model 3 Legal Prior-Case Retrieval (BM25 + Cross-Encoder)

This replaces the earlier prefix-only bi-encoder experiment. The earlier run trained only 147,456 prefix parameters (0.19% of the 78.8M model), for two epochs, using random negatives and mean-pooled embeddings. Its training loss did not improve and Dev Micro-F1 was 0.01334.

This notebook uses the exact Phase-1 Model 3 checkpoint and tokenizer, but changes the retrieval formulation:
1. **BM25 hard-negative mining** from the permitted Train candidate pool.
2. **Cross-encoder ranking**: jointly encodes each query–candidate pair, so attention can model legal relevance across both cases.
3. **Partial fine-tuning** of the final three encoder blocks and final normalization, plus a learned scoring head.
4. **BM25 candidate generation + neural reranking** for Dev and Test.
5. Dev reports both BM25-only and neural-reranked Micro-F1@7. The best ranking is selected by the Dev metric; Test labels remain private.

All outputs are saved directly in `/kaggle/working/phase7_experiments/`. Designed for Kaggle T4 with mixed precision. Training may take substantially longer than the earlier 1.2-minute prefix run. It is not possible to guarantee a score in advance; Dev results determine whether the neural reranker actually helps.

In [1]:
# EXPERIMENT 1 — Model 3 + Prefix Tuning (CS781 Phase 7)
# Self-contained Kaggle notebook. No Phase-1 pretraining is run here.

import os, sys, glob, json, time, math, random, re, heapq, gc, types, subprocess, zipfile
from pathlib import Path
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download

SEED = 731
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def stamp():
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d %H:%M:%S IST")

print("=" * 78)
print("PHASE 7 | EXPERIMENT 1 | MODEL 3 + PREFIX TUNING")
print("Started:", stamp())
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("=" * 78)

# ---------------------- Find user's attached Model 3 checkpoint/tokenizer ----------------------
all_pt = glob.glob("/kaggle/input/**/*.pt", recursive=True)
best_candidates = [
    p for p in all_pt
    if "encdec_full_best" in os.path.basename(p).lower()
]
if not best_candidates:
    best_candidates = [
        p for p in all_pt
        if "best" in os.path.basename(p).lower()
        and ("model3" in p.lower() or "encdec" in p.lower())
    ]
if not best_candidates:
    raise FileNotFoundError(
        "Could not find Model 3 best checkpoint under /kaggle/input. "
        "Attach your Phase-1 Model 3 best.pt dataset."
    )

# Prefer the exact file used in the uploaded Phase-1 notebook.
preferred = [
    p for p in best_candidates
    if "phase1-model3-ckpts" in p.lower()
]
CKPT_PATH = sorted(preferred or best_candidates)[0]

tok_candidates = glob.glob("/kaggle/input/**/tokenizer (1).json", recursive=True)
if not tok_candidates:
    raise FileNotFoundError(
        "Could not find tokenizer.json under /kaggle/input. "
        "Attach the matching Model 3 tokenizer dataset."
    )
# Prefer a tokenizer mounted alongside the Model 3 checkpoint or explicitly named Model 3.
tok_preferred = [
    p for p in tok_candidates
    if ("model3" in p.lower() or "encdec" in p.lower())
]
if not tok_preferred:
    # If only one tokenizer is attached, it is unambiguous.
    if len(tok_candidates) == 1:
        tok_preferred = tok_candidates
    else:
        print("Available tokenizer files:")
        for p in tok_candidates:
            print(" -", p)
        raise RuntimeError(
            "Multiple tokenizer.json files found and none is clearly Model 3. "
            "Set TOKENIZER_PATH manually to the Model 3 tokenizer."
        )
TOKENIZER_PATH = sorted(tok_preferred)[0]

print("Checkpoint:", CKPT_PATH)
print("Tokenizer:", TOKENIZER_PATH)

# ---------------------- Phase 7 dataset download/load ----------------------
REPO_ID = "Exploration-Lab/CS-781-Capstone"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face token loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = None
    print("No HF_TOKEN secret found; trying dataset download without authentication.")

def load_hf_json(relative_path):
    local = hf_hub_download(
        repo_id=REPO_ID,
        filename=relative_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    with open(local, "r", encoding="utf-8") as f:
        return json.load(f)

print("\nLoading Phase 7 data:", stamp())
train_queries = load_hf_json("Phase_7_PCR/Train/train.json")
train_candidates = load_hf_json("Phase_7_PCR/Train/train_candidates.json")
dev_queries = load_hf_json("Phase_7_PCR/Dev/dev.json")
dev_candidates = load_hf_json("Phase_7_PCR/Dev/dev_candidates.json")
test_queries = load_hf_json("Phase_7_PCR/Test/test.json")
test_candidates = load_hf_json("Phase_7_PCR/Test/test_candidates.json")

print(
    f"Train: {len(train_queries):,} queries | {len(train_candidates):,} candidates"
)
print(
    f"Dev:   {len(dev_queries):,} queries | {len(dev_candidates):,} candidates"
)
print(
    f"Test:  {len(test_queries):,} queries | {len(test_candidates):,} candidates"
)

# ---------------------- Experiment configuration ----------------------
D_MODEL = 512
N_ENCODER_LAYERS = 9
N_DECODER_LAYERS = 9
N_HEADS = 8
FFN_DIM = 2048
DROPOUT = 0.0
MAX_ENCODER_POSITIONS = 2048
MAX_DECODER_POSITIONS = 1536

# Retrieval-focused configuration.
# The model is trained as a cross-encoder on BM25-mined hard negatives.
PAIR_MAX_LEN = 320
QUERY_TOKENS = 112
CANDIDATE_TOKENS = 204
HARD_NEGATIVE_POOL = 80
TRAIN_NEGATIVES = 5
RERANK_TOP_N = 100
PAIR_BATCH_SIZE = 12
GRAD_ACCUM_STEPS = 1
EPOCHS = 4
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
EVAL_BATCH_SIZE = 12
TOP_K = 7
UNFREEZE_LAST_N = 3
SEED = 731

OUTPUT_DIR = Path("/kaggle/working/phase7_experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nConfiguration:")
print({
    "pair_max_len": PAIR_MAX_LEN,
    "query_tokens": QUERY_TOKENS,
    "candidate_tokens": CANDIDATE_TOKENS,
    "hard_negative_pool": HARD_NEGATIVE_POOL,
    "train_negatives": TRAIN_NEGATIVES,
    "rerank_top_n": RERANK_TOP_N,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "unfrozen_encoder_layers": UNFREEZE_LAST_N,
    "started_at": stamp()
})


PHASE 7 | EXPERIMENT 1 | MODEL 3 + PREFIX TUNING
Started: 2026-11-06 00:51:06 IST
Device: cuda
GPU: Tesla T4
Checkpoint: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt
Tokenizer: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json
Hugging Face token loaded from Kaggle Secrets.

Loading Phase 7 data: 2026-11-06 00:51:07 IST


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (2 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Train: 827 queries | 4,320 candidates
Dev:   118 queries | 1,023 candidates
Test:  237 queries | 1,727 candidates

Configuration:
{'pair_max_len': 320, 'query_tokens': 112, 'candidate_tokens': 204, 'hard_negative_pool': 80, 'train_negatives': 5, 'rerank_top_n': 100, 'epochs': 4, 'learning_rate': 1.5e-05, 'unfrozen_encoder_layers': 3, 'started_at': '2026-11-06 00:51:19 IST'}


In [2]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

# ---------------------- Load the exact Phase-1 Model 3 tokenizer ----------------------
tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
print("Tokenizer vocabulary:", len(tokenizer))
print("Special IDs:", {
    "pad": tokenizer.pad_token_id,
    "unk": tokenizer.unk_token_id,
    "bos": tokenizer.bos_token_id,
    "eos": tokenizer.eos_token_id
})
assert len(tokenizer) == 24576, (
    f"Expected Model 3 vocab 24576, got {len(tokenizer)}. "
    "Check that the selected tokenizer belongs to Model 3."
)


Tokenizer class ready.
Tokenizer vocabulary: 24576
Special IDs: {'pad': 0, 'unk': 1, 'bos': 2, 'eos': 3}


In [3]:
# Cell 9 — ~79M Encoder–Decoder Transformer (SDPA, cross-attention, decode KV cache)
# v2 changes vs v1:
#   * token embeddings are scaled by sqrt(d_model) before the sinusoidal positions are added
#     (v1 added unit-scale positions to 0.02-std embeddings => positions drowned the tokens)
#   * encoder positions are derived from the padding mask (so padded / doc-masked rows get positions 0..n-1)
#   * decoder self-attention handles "cache + several new tokens" correctly (needed for prompt priming)
#   * residual-branch init uses the true branch count (3 per decoder layer)
#   * build_model() so Cell 11 always starts from fresh weights (no more "rerun Cell 9" step)
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class EncoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, key_valid=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        attn_mask = None
        if key_valid is not None:
            attn_mask = key_valid[:, None, None, :]  # True = valid encoder key

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class DecoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        mask, is_causal = None, True
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
            if T == 1:
                is_causal = False                      # the new token may see every cached key
            else:
                S = k.size(2)
                P = S - T                              # query i sits at global position P+i
                mask = torch.ones(T, S, device=x.device).tril(diagonal=P).bool()
                is_causal = False

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.kv_proj = nn.Linear(d_model, 2 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, memory, memory_valid=None, cached_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if cached_kv is None:
            S = memory.size(1)
            kv = self.kv_proj(memory).view(B, S, 2, self.n_heads, self.head_dim)
            k, v = kv.unbind(dim=2)
            k, v = k.transpose(1, 2), v.transpose(1, 2)
        else:
            k, v = cached_kv

        attn_mask = None
        if memory_valid is not None:
            attn_mask = memory_valid[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = EncoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_valid=None):
        x = x + self.drop(self.attn(self.ln1(x), key_valid=key_valid))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = DecoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads, dropout)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(
        self, x, memory, memory_valid=None,
        past_self_kv=None, cached_cross_kv=None, use_cache=False
    ):
        a, new_self = self.self_attn(
            self.ln1(x), past_kv=past_self_kv, use_cache=use_cache
        )
        x = x + self.drop(a)

        c, new_cross = self.cross_attn(
            self.ln2(x),
            memory,
            memory_valid=memory_valid,
            cached_kv=cached_cross_kv,
            use_cache=use_cache,
        )
        x = x + self.drop(c)
        x = x + self.ff(self.ln3(x))
        return x, new_self, new_cross


class EncoderDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_encoder_layers, n_decoder_layers,
        n_heads, ffn_dim, dropout, max_encoder_positions, max_decoder_positions
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.emb_scale = math.sqrt(d_model)
        self.max_encoder_positions = max_encoder_positions
        self.max_decoder_positions = max_decoder_positions

        # Shared embedding across encoder input, decoder input and output projection.
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.enc_pos = SinusoidalPositionalEncoding(d_model, max_encoder_positions)
        self.dec_pos = SinusoidalPositionalEncoding(d_model, max_decoder_positions)
        self.drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_encoder_layers)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_decoder_layers)
        ])
        self.enc_ln = nn.LayerNorm(d_model)
        self.dec_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-2 style depth scaling: std = 0.02 / sqrt(#residual branches)
        enc_scaled = 0.02 / math.sqrt(2 * n_encoder_layers)   # attn + ffn
        dec_scaled = 0.02 / math.sqrt(3 * n_decoder_layers)   # self-attn + cross-attn + ffn
        for block in self.encoder:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=enc_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=enc_scaled)
        for block in self.decoder:
            nn.init.normal_(block.self_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.cross_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=dec_scaled)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, input_ids, key_valid=None, position_ids=None):
        B, S = input_ids.shape
        if S > self.max_encoder_positions:
            raise ValueError(f"encoder length {S} > {self.max_encoder_positions}")

        if position_ids is None:
            if key_valid is None:
                position_ids = torch.arange(S, device=input_ids.device)[None, :].expand(B, S)
            else:
                # valid tokens always get positions 0..n-1, wherever the padding / masked tokens sit
                position_ids = (key_valid.long().cumsum(1) - 1).clamp(min=0)

        x = self.token_emb(input_ids) * self.emb_scale \
            + self.enc_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)
        for block in self.encoder:
            x = block(x, key_valid=key_valid)
        return self.enc_ln(x)

    def decode_hidden(
        self, decoder_ids, memory, memory_valid=None,
        past_self_key_values=None, cross_key_values=None,
        use_cache=False, position_ids=None
    ):
        B, T = decoder_ids.shape
        past_len = 0
        if past_self_key_values is not None:
            past_len = past_self_key_values[0][0].size(2)

        if past_len + T > self.max_decoder_positions:
            raise ValueError(
                f"decoder length {past_len + T} > {self.max_decoder_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=decoder_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(decoder_ids) * self.emb_scale \
            + self.dec_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_self, new_cross = ([] if use_cache else None), ([] if use_cache else None)

        for i, block in enumerate(self.decoder):
            ps = None if past_self_key_values is None else past_self_key_values[i]
            ck = None if cross_key_values is None else cross_key_values[i]
            x, s_kv, c_kv = block(
                x, memory, memory_valid=memory_valid,
                past_self_kv=ps, cached_cross_kv=ck, use_cache=use_cache
            )
            if use_cache:
                new_self.append(s_kv)
                new_cross.append(c_kv)

        return self.dec_ln(x), new_self, new_cross

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_model():
    return EncoderDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_encoder_layers=N_ENCODER_LAYERS,
        n_decoder_layers=N_DECODER_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_encoder_positions=MAX_ENCODER_POSITIONS,
        max_decoder_positions=MAX_DECODER_POSITIONS,
    ).to(DEVICE)

# ---- instantiate
model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print("Architecture: 9-layer bidirectional encoder + 9-layer causal cross-attention decoder")
assert 75_000_000 <= n_params <= 85_000_000, n_params


Model parameters: 78.79M
Architecture: 9-layer bidirectional encoder + 9-layer causal cross-attention decoder


In [4]:

# ---------------------- BM25 index + hard-negative mining ----------------------
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

def record_text(record):
    value = record.get("text", [])
    if isinstance(value, list):
        return " ".join(str(x) for x in value)
    return str(value)

def bm25_clean(text):
    # Keep legal section numbers and alphanumeric case references.
    return re.sub(r"\s+", " ", str(text)).strip()

class BM25Index:
    def __init__(self, docs, k1=1.35, b=0.72, max_features=300000):
        self.ids = list(docs.keys())
        self.texts = [bm25_clean(record_text(docs[x])) for x in self.ids]
        self.vectorizer = CountVectorizer(
            lowercase=True, stop_words="english",
            token_pattern=r"(?u)\b[a-zA-Z0-9][a-zA-Z0-9./_-]{1,}\b",
            ngram_range=(1, 2), max_features=max_features,
            min_df=1, binary=False, dtype=np.float32
        )
        counts = self.vectorizer.fit_transform(self.texts).tocsr()
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.avgdl = float(self.doc_len.mean()) if len(self.doc_len) else 1.0
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(np.float32)
        n = len(self.ids)
        self.idf = np.log(1.0 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        # Build BM25 weights directly on nonzero entries; never densify the
        # document-term matrix (which can exceed several GB for legal corpora).
        row_ids = np.repeat(np.arange(counts.shape[0]), np.diff(counts.indptr))
        tf = counts.data.astype(np.float32, copy=False)
        norm = k1 * (1.0 - b + b * self.doc_len[row_ids] / max(self.avgdl, 1.0))
        weighted = self.idf[counts.indices] * (tf * (k1 + 1.0) / (tf + norm))
        self.matrix = sparse.csr_matrix(
            (weighted.astype(np.float32), counts.indices.copy(), counts.indptr.copy()),
            shape=counts.shape
        )
        self.k1, self.b = k1, b
        self.id_to_row = {x:i for i,x in enumerate(self.ids)}

    def search(self, query_record, top_n=100, exclude_id=None):
        q = self.vectorizer.transform([bm25_clean(record_text(query_record))])
        scores = np.asarray((self.matrix @ q.T).toarray()).ravel()
        if exclude_id in self.id_to_row:
            scores[self.id_to_row[exclude_id]] = -1e9
        k = min(top_n, len(scores))
        idx = np.argpartition(-scores, k-1)[:k]
        idx = idx[np.argsort(-scores[idx])]
        return [(self.ids[i], float(scores[i])) for i in idx]

print("Building BM25 index over Train candidate pool:", stamp())
bm25_train = BM25Index(train_candidates)
print("Train BM25 vocabulary:", len(bm25_train.vectorizer.vocabulary_))

# Cache BM25 hard-negative rankings per query. All labels come only from Train.
train_examples = []
missing_positive = 0
for qid, query in train_queries.items():
    positives = [x for x in query.get("relevant_candidates", [])
                 if x in train_candidates and x != qid]
    if not positives:
        missing_positive += 1
        continue
    ranked = bm25_train.search(query, top_n=HARD_NEGATIVE_POOL, exclude_id=qid)
    positive_set = set(positives)
    hard_negs = [cid for cid, _ in ranked if cid not in positive_set]
    if len(hard_negs) < TRAIN_NEGATIVES:
        hard_negs += [cid for cid in train_candidates
                      if cid not in positive_set and cid != qid and cid not in hard_negs]
    train_examples.append({
        "qid": qid, "positive_ids": positives,
        "hard_negative_ids": hard_negs[:max(TRAIN_NEGATIVES*3, 20)]
    })
print(f"Train query groups: {len(train_examples)} | without usable positives: {missing_positive}")

# ---------------------- Build and load the exact Phase-1 Model 3 ----------------------
model = build_model()
checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
if "model" not in checkpoint:
    raise KeyError(f"Checkpoint lacks 'model'. Keys: {list(checkpoint.keys())}")
model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(DEVICE)

# Train the final encoder blocks + final norm; freeze the rest and the decoder.
for p in model.parameters():
    p.requires_grad = False
for block in model.encoder[-UNFREEZE_LAST_N:]:
    for p in block.parameters():
        p.requires_grad = True
for p in model.enc_ln.parameters():
    p.requires_grad = True

class PairScorer(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.score = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )
    def forward(self, input_ids, valid):
        h = self.backbone.encode(input_ids, key_valid=valid)
        mask = valid.unsqueeze(-1).to(h.dtype)
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.score(self.dropout(pooled.float())).squeeze(-1)

pair_model = PairScorer(model).to(DEVICE)
trainable = sum(p.numel() for p in pair_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} / {sum(p.numel() for p in pair_model.parameters()):,}")

def encode_side(record, limit):
    ids = tokenizer.encode(record_text(record))
    if not ids:
        ids = [tokenizer.unk_token_id]
    if len(ids) > limit:
        head = int(limit * 0.72)
        ids = ids[:head] + ids[-(limit-head):]
    return ids

def make_pair_ids(query_record, candidate_record):
    q = encode_side(query_record, QUERY_TOKENS)
    c = encode_side(candidate_record, CANDIDATE_TOKENS)
    ids = [tokenizer.bos_token_id] + q + [tokenizer.eos_token_id] + c + [tokenizer.eos_token_id]
    return ids[:PAIR_MAX_LEN]

def collate_pairs(pairs):
    rows = [make_pair_ids(q, c) for q, c in pairs]
    width = max(len(x) for x in rows)
    x = torch.full((len(rows), width), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(rows), width), dtype=torch.bool)
    for i, row in enumerate(rows):
        x[i, :len(row)] = torch.tensor(row, dtype=torch.long)
        mask[i, :len(row)] = True
    return x.to(DEVICE), mask.to(DEVICE)

# One positive plus BM25 hard negatives per query. Resample a positive each epoch.
optimizer = torch.optim.AdamW(
    [p for p in pair_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
steps_per_epoch = math.ceil(len(train_examples) / 2)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

def make_training_group(ex):
    q = train_queries[ex["qid"]]
    pos_id = random.choice(ex["positive_ids"])
    neg_ids = random.sample(ex["hard_negative_ids"],
                            k=min(TRAIN_NEGATIVES, len(ex["hard_negative_ids"])))
    pairs = [(q, train_candidates[pos_id])]
    pairs.extend((q, train_candidates[cid]) for cid in neg_ids)
    return pairs

print("\nCross-encoder training started:", stamp())
train_start = time.time()
best_dev_f1 = -1.0
history = []
pair_model.train()

# Two query groups per optimizer step; flattened pair batch stays T4-friendly.
QUERY_GROUP_BATCH = 2
for epoch in range(EPOCHS):
    random.shuffle(train_examples)
    running, steps = 0.0, 0
    for start in range(0, len(train_examples), QUERY_GROUP_BATCH):
        groups = train_examples[start:start+QUERY_GROUP_BATCH]
        flat_pairs, group_sizes = [], []
        for ex in groups:
            gp = make_training_group(ex)
            flat_pairs.extend(gp)
            group_sizes.append(len(gp))
        x, valid = collate_pairs(flat_pairs)
        pair_model.train()
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(DEVICE.type == "cuda")):
            logits = pair_model(x, valid)
        # Per-query listwise loss: positive is always first in each group.
        losses, offset = [], 0
        for size in group_sizes:
            group_logits = logits[offset:offset+size].float().unsqueeze(0)
            losses.append(F.cross_entropy(group_logits, torch.zeros(1, dtype=torch.long, device=DEVICE)))
            offset += size
        loss = torch.stack(losses).mean()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in pair_model.parameters() if p.requires_grad], 1.0
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += float(loss.detach())
        steps += 1
        if steps % 25 == 0 or start + QUERY_GROUP_BATCH >= len(train_examples):
            elapsed = time.time() - train_start
            print(f"[{stamp()}] epoch {epoch+1}/{EPOCHS} step {steps}/{steps_per_epoch} "
                  f"loss={running/max(steps,1):.4f} elapsed={elapsed/60:.1f}m", flush=True)
    epoch_loss = running / max(steps, 1)
    history.append(epoch_loss)
    torch.save({
        "model_state": pair_model.state_dict(),
        "epoch": epoch+1, "loss": epoch_loss,
        "base_checkpoint": CKPT_PATH, "tokenizer_path": TOKENIZER_PATH,
        "config": {"epochs": EPOCHS, "pair_max_len": PAIR_MAX_LEN,
                   "query_tokens": QUERY_TOKENS, "candidate_tokens": CANDIDATE_TOKENS,
                   "unfreeze_last_n": UNFREEZE_LAST_N}
    }, OUTPUT_DIR / f"phase7_crossencoder_epoch{epoch+1}.pt")
    print(f"Epoch {epoch+1} mean loss={epoch_loss:.5f} | saved checkpoint")
print("Training finished:", stamp(), "| minutes:", round((time.time()-train_start)/60, 2))
print("Epoch losses:", history)


Building BM25 index over Train candidate pool: 2026-11-06 00:51:22 IST
Train BM25 vocabulary: 300000
Train query groups: 817 | without usable positives: 10
Trainable parameters: 9,589,761 / 78,924,289

Cross-encoder training started: 2026-11-06 00:53:03 IST
[2026-11-06 00:53:14 IST] epoch 1/4 step 25/409 loss=1.7911 elapsed=0.2m
[2026-11-06 00:53:25 IST] epoch 1/4 step 50/409 loss=1.7912 elapsed=0.4m
[2026-11-06 00:53:34 IST] epoch 1/4 step 75/409 loss=1.7912 elapsed=0.5m
[2026-11-06 00:53:43 IST] epoch 1/4 step 100/409 loss=1.7914 elapsed=0.7m
[2026-11-06 00:53:52 IST] epoch 1/4 step 125/409 loss=1.7916 elapsed=0.8m
[2026-11-06 00:54:01 IST] epoch 1/4 step 150/409 loss=1.7917 elapsed=1.0m
[2026-11-06 00:54:11 IST] epoch 1/4 step 175/409 loss=1.7916 elapsed=1.1m
[2026-11-06 00:54:22 IST] epoch 1/4 step 200/409 loss=1.7917 elapsed=1.3m
[2026-11-06 00:54:31 IST] epoch 1/4 step 225/409 loss=1.7915 elapsed=1.5m
[2026-11-06 00:54:42 IST] epoch 1/4 step 250/409 loss=1.7915 elapsed=1.6m
[2026

In [5]:

# ---------------------- Evaluation helpers: BM25 retrieve + neural rerank ----------------------
@torch.no_grad()
def score_pairs(pair_list, batch_size=EVAL_BATCH_SIZE):
    pair_model.eval()
    out = []
    for start in range(0, len(pair_list), batch_size):
        batch = pair_list[start:start+batch_size]
        x, valid = collate_pairs(batch)
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(DEVICE.type == "cuda")):
            logits = pair_model(x, valid)
        out.extend(logits.float().cpu().numpy().tolist())
    return np.asarray(out, dtype=np.float32)

def retrieve_and_rerank(queries, candidates, split_name, top_n=RERANK_TOP_N):
    print(f"\nBuilding {split_name} BM25 index:", stamp())
    index = BM25Index(candidates)
    output = {}
    bm25_only = {}
    for qi, (qid, qrec) in enumerate(queries.items(), 1):
        ranked = index.search(qrec, top_n=top_n, exclude_id=qid)
        bm25_only[qid] = [cid for cid, _ in ranked[:TOP_K]]
        pair_list = [(qrec, candidates[cid]) for cid, _ in ranked]
        neural = score_pairs(pair_list)
        # Rank by cross-encoder score; lexical score is used as a small calibrated tie-break.
        lexical = np.asarray([s for _, s in ranked], dtype=np.float32)
        if len(lexical):
            lexical = (lexical - lexical.mean()) / (lexical.std() + 1e-6)
        neural = (neural - neural.mean()) / (neural.std() + 1e-6) if len(neural) else neural
        fused = neural + 0.12 * lexical
        order = np.argsort(-fused)
        output[qid] = [ranked[j][0] for j in order[:TOP_K]]
        if qi % 10 == 0 or qi == len(queries):
            print(f"[{stamp()}] {split_name}: reranked {qi}/{len(queries)} queries", flush=True)
    return output, bm25_only

def official_metrics(queries, predictions):
    tp = fp = fn = 0
    for qid, q in queries.items():
        gold = set(q.get("relevant_candidates", []))
        pred = set(predictions.get(qid, []))
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    return {"micro_f1_at_7": f1, "micro_precision_at_7": p,
            "micro_recall_at_7": r, "TP": tp, "FP": fp, "FN": fn}

print("Dev evaluation started:", stamp())
dev_predictions, dev_bm25_predictions = retrieve_and_rerank(
    dev_queries, dev_candidates, "Dev"
)
dev_metrics = official_metrics(dev_queries, dev_predictions)
bm25_metrics = official_metrics(dev_queries, dev_bm25_predictions)
print("\nDEV RESULTS")
print("BM25 baseline:", json.dumps(bm25_metrics, indent=2))
print("Cross-encoder rerank:", json.dumps(dev_metrics, indent=2))
(OUTPUT_DIR / "dev_predictions_crossencoder.json").write_text(
    json.dumps(dev_predictions, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "dev_metrics_crossencoder.json").write_text(
    json.dumps({"bm25": bm25_metrics, "cross_encoder_rerank": dev_metrics,
                "training_loss": history}, indent=2), encoding="utf-8"
)


Dev evaluation started: 2026-11-06 01:03:52 IST

Building Dev BM25 index: 2026-11-06 01:03:52 IST
[2026-11-06 01:05:02 IST] Dev: reranked 10/118 queries
[2026-11-06 01:05:54 IST] Dev: reranked 20/118 queries
[2026-11-06 01:06:48 IST] Dev: reranked 30/118 queries
[2026-11-06 01:07:48 IST] Dev: reranked 40/118 queries
[2026-11-06 01:08:44 IST] Dev: reranked 50/118 queries
[2026-11-06 01:09:36 IST] Dev: reranked 60/118 queries
[2026-11-06 01:10:31 IST] Dev: reranked 70/118 queries
[2026-11-06 01:11:29 IST] Dev: reranked 80/118 queries
[2026-11-06 01:12:21 IST] Dev: reranked 90/118 queries
[2026-11-06 01:13:18 IST] Dev: reranked 100/118 queries
[2026-11-06 01:14:08 IST] Dev: reranked 110/118 queries
[2026-11-06 01:15:04 IST] Dev: reranked 118/118 queries

DEV RESULTS
BM25 baseline: {
  "micro_f1_at_7": 0.330183435241801,
  "micro_precision_at_7": 0.3595641646489104,
  "micro_recall_at_7": 0.3052415210688592,
  "TP": 297,
  "FP": 529,
  "FN": 676
}
Cross-encoder rerank: {
  "micro_f1_at_7":

527

In [6]:

# ---------------------- Test inference + official-format ZIP ----------------------
print("Test inference started:", stamp())
test_reranked_predictions, test_bm25_predictions = retrieve_and_rerank(
    test_queries, test_candidates, "Test"
)

# Select the approach that actually scored better on labelled Dev.
# This prevents submitting a neural reranker that hurts the lexical baseline.
if dev_metrics["micro_f1_at_7"] >= bm25_metrics["micro_f1_at_7"]:
    test_predictions = test_reranked_predictions
    selected_method = "BM25 + Model 3 cross-encoder reranking"
else:
    test_predictions = test_bm25_predictions
    selected_method = "BM25-only (Dev cross-encoder reranking did not improve F1)"

print("Selected Test method:", selected_method)
assert set(test_predictions) == set(test_queries)
for qid, ids in test_predictions.items():
    assert len(ids) == 7 and len(set(ids)) == 7
    assert all(cid in test_candidates for cid in ids)

prediction_path = OUTPUT_DIR / "prediction.json"
prediction_path.write_text(
    json.dumps(test_predictions, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)
with open(prediction_path, "r", encoding="utf-8") as f:
    check = json.load(f)
assert check == test_predictions

zip_path = OUTPUT_DIR / "phase7_crossencoder_submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(prediction_path, arcname="prediction.json")
with zipfile.ZipFile(zip_path, "r") as z:
    assert z.namelist() == ["prediction.json"]
    assert json.loads(z.read("prediction.json").decode("utf-8")) == test_predictions

print("\nSUBMISSION READY")
print("Prediction:", prediction_path)
print("ZIP:", zip_path)
print("Queries:", len(test_predictions), "| candidates:", len(test_candidates))
print("Dev BM25 F1:", round(bm25_metrics["micro_f1_at_7"], 6))
print("Dev reranker F1:", round(dev_metrics["micro_f1_at_7"], 6))
print("Selected method:", selected_method)
print("Test relevance labels are private; no local Test F1 is available.")
print("Finished:", stamp())


Test inference started: 2026-11-06 01:15:04 IST

Building Test BM25 index: 2026-11-06 01:15:04 IST
[2026-11-06 01:16:16 IST] Test: reranked 10/237 queries
[2026-11-06 01:16:56 IST] Test: reranked 20/237 queries
[2026-11-06 01:17:28 IST] Test: reranked 30/237 queries
[2026-11-06 01:17:56 IST] Test: reranked 40/237 queries
[2026-11-06 01:18:24 IST] Test: reranked 50/237 queries
[2026-11-06 01:19:03 IST] Test: reranked 60/237 queries
[2026-11-06 01:19:36 IST] Test: reranked 70/237 queries
[2026-11-06 01:20:18 IST] Test: reranked 80/237 queries
[2026-11-06 01:20:56 IST] Test: reranked 90/237 queries
[2026-11-06 01:21:27 IST] Test: reranked 100/237 queries
[2026-11-06 01:22:01 IST] Test: reranked 110/237 queries
[2026-11-06 01:22:33 IST] Test: reranked 120/237 queries
[2026-11-06 01:23:09 IST] Test: reranked 130/237 queries
[2026-11-06 01:23:41 IST] Test: reranked 140/237 queries
[2026-11-06 01:24:21 IST] Test: reranked 150/237 queries
[2026-11-06 01:24:58 IST] Test: reranked 160/237 querie